In [2]:
import pandas as pd
import numpy as np

In [ ]:
enbs_paths = "03_output/03_enbs"

# main results
df_a_val = pd.read_csv(f"{enbs_paths}/01_a_val.csv")
df_price = pd.read_csv(f"{enbs_paths}/02_drug_price.csv")
df_sample_size = pd.read_csv(f"{enbs_paths}/03_sample_size.csv")

In [ ]:
# !pip install -q lets-plot
from lets_plot import *
LetsPlot.setup_html()

# Add font family configuration
times_new_roman_theme = theme(
    title=element_text(size=12, family="Times New Roman"),
    axis_text_x=element_text(size=10, family="Times New Roman"),
    axis_text_y=element_text(size=10, family="Times New Roman"),
    legend_text=element_text(size=12, family="Times New Roman"),
    # legend_position="right",  # Move legend to the bottom
    # legend_title=element_blank(),  # Remove color legend title
    legend_title=element_text(size=12, family="Times New Roman"),
    plot_title=element_text(size=14, family="Times New Roman"),
    plot_subtitle=element_text(size=12, family="Times New Roman"),
    axis_title_x=element_text(size=12, family="Times New Roman"),
    axis_title_y=element_text(size=12, family="Times New Roman")
)

In [5]:
# draw the main results
df_evpi_evsi = df_a_val.melt(id_vars=["a_value"], value_vars=["EVPI", "EVSI"], var_name="Metric", value_name="Value")

# if Metric is EVPI, change to EVPLC; if EVSI, EVSLC
df_evpi_evsi["Metric"] = df_evpi_evsi["Metric"].apply(lambda x: "EVPLC" if x == "EVPI" else "EVSLC")

# select rows with a_value >= 0.5
df_evpi_evsi = df_evpi_evsi[df_evpi_evsi["a_value"] >= 0.5]

plot_a_val_evpi_evsi = (
    ggplot(df_evpi_evsi, aes(x="a_value", y="Value", shape="Metric", group="Metric")) +
    geom_point(size = 3.2) +
    geom_line() +

    scale_x_continuous(breaks=df_a_val["a_value"].unique(), format="{.1f}") +
    scale_y_continuous(format="${.1f}") +

    labs(
        x = "Power Parameter",
        y = "Population-level EVPLC/EVSLC ($billion)",
        shape = ""
    ) +
    theme_bw() +
    times_new_roman_theme +
    theme(
        legend_position=[0.9, 0.8],
        legend_background=element_blank()
    )
)

plot_a_val_evpi_evsi

In [6]:
df_a_val_enbs = df_a_val[df_a_val["a_value"] >= 0.5]

plot_a_val_enbs = (
    ggplot(df_a_val_enbs, aes(x="a_value", y="ENBS")) +
    geom_point(size = 3.2) +
    geom_line() +
    scale_x_continuous(breaks=df_a_val["a_value"].unique(), format="{.1f}") +
    scale_y_continuous(breaks = [0.7, 0.8, 0.9, 1.0], format="${.1f}", limits=[0.7, 1.05]) +
    # geom_hline(yintercept=0, linetype='dashed', color='#434343') +
    labs(
        x = "Power Parameter",
        y = "Population-level ENBSLC ($billion)"
    ) +
    theme_bw() +
    times_new_roman_theme
)

plot_a_val_enbs

In [7]:
plot_list=[
    plot_a_val_evpi_evsi + ggtitle("(A): Population-level EVPLC/EVSLC under different uncertainty levels") + theme(title=element_text(face="bold")),
    plot_a_val_enbs + ggtitle("(B): Population-level ENBSLC under different uncertainty levels") + theme(title=element_text(face="bold"))
]

plot_1 = gggrid(plot_list, ncol=1)

plot_1

In [8]:
# add two columns:
## "intro_drug": 200 * price_drug + 6992.538
## "maintain_drug": 200 * price_drug + 6965.7
df_price["intro_drug"] = 200 * df_price["price_drug"] + 6992.538
df_price["maintain_drug"] = 200 * df_price["price_drug"] + 6965.7
df_price['drug_cost_per_cycle'] = ((df_price['intro_drug']*4 + df_price['maintain_drug']*32)/36)/1000

df_price.to_csv(f"{enbs_paths}/02_drug_price.csv", index=False)

In [9]:
df_price = df_price[~df_price['price_drug'].between(14.6, 14.9)]

# Check with drug price
plot_price_enbs = (
    ggplot(df_price, aes(x="price_drug", y="ENBS", shape = "a_value", group = "a_value")) +
    geom_point(size = 3.2) +
    geom_line() +
    scale_x_continuous(format="${.1f}") +
    scale_y_continuous(format="${.1f}") +
    geom_hline(yintercept=0, linetype='dashed', color='#434343') +
    scale_shape_manual(values={
        1.0: 17,   # Circle
        0.75: 15,  # Triangle
        0.5: 16    # Cross
    }) +
    labs(
        x = "Price of Sintilimab per Dose",
        y = "Population-level ENBSLC ($billion)",
        shape = "Power Parameter"
    ) +
    theme_bw() +
    times_new_roman_theme
)

plot_price_enbs

In [10]:
# Check with per-cycle price
plot_price_cycle_enbs = (
    ggplot(df_price, aes(x="drug_cost_per_cycle", y="ENBS", shape = "a_value", group = "a_value")) +
    geom_point(size = 3.2) +
    geom_line() +
    scale_x_continuous(format="${.1f}") +
    scale_y_continuous(format="${.1f}") +
    geom_hline(yintercept=0, linetype='dashed', color='#434343') +
    scale_shape_manual(values={
        1.0: 17,   # Circle
        0.75: 15,  # Triangle
        0.5: 16    # Cross
    }) +
    labs(
        x = "Average Treatment Cost per Cycle ($thousand)",
        y = "Population-level ENBSLC ($billion)",
        shape = "Power Parameter"
    ) +
    theme_bw() +
    times_new_roman_theme
)

plot_price_cycle_enbs

In [11]:
df_sample_size = df_sample_size[~df_sample_size['sample_size_new'].between(601, 650)]
# Check with sample size increasing for EVSI
plot_sample_size_evsi = (
    ggplot(df_sample_size, aes(x="sample_size_new", y="ENBS", shape = "a_value", group = "a_value")) +
    geom_point(size = 3.2) +
    geom_line() +
    scale_x_continuous(breaks=np.arange(300, 1100, 100)) +
    scale_y_continuous(format="${.1f}") +
    # geom_hline(yintercept=0, linetype='dashed', color='#434343') +
    scale_shape_manual(values={
        1.0: 17,   # Circle
        0.75: 15,  # Triangle
        0.5: 16    # Cross
    }) +
    labs(
        x = "Sample Size for the Local Confirmatory Trial",
        y = "Population-level ENBSLC ($billion)",
        shape = "Power Parameter"
    ) +
    theme_bw() +
    times_new_roman_theme
)

plot_sample_size_evsi

In [12]:
plot_list_2=[
    plot_sample_size_evsi +
    ggtitle("(A): Varying sample sizes under different uncertainty levels") +
    theme(title=element_text(face="bold")),
    plot_price_enbs +
    ggtitle("(B): Varying per-dose price under different uncertainty levels" ) +
    theme(title=element_text(face="bold"))
]

plot_2 = gggrid(plot_list_2, ncol=1)

plot_2

In [13]:
plot_list_3=[
    plot_sample_size_evsi +
    ggtitle("(A): Varying sample sizes under different uncertainty levels") +
    theme(title=element_text(face="bold")),
    plot_price_cycle_enbs +
    ggtitle("(B): Varying average per-cycle treatment cost under different uncertainty levels" ) +
    theme(title=element_text(face="bold"))
]

plot_3 = gggrid(plot_list_3, ncol=1)

plot_3

In [ ]:
# !pip install -q CairoSVG
ggsave(plot=plot_a_val_evpi_evsi, filename=f"{enbs_paths}/01_evpi_evsi.pdf", dpi=800, w=4*4, h=3*4, unit='in')
ggsave(plot=plot_a_val_enbs, filename=f"{enbs_paths}/02_enbs.pdf", dpi=800, w=4*4, h=3*4, unit='in')
ggsave(plot=plot_sample_size_evsi, filename=f"{enbs_paths}/03_sample_size.pdf", dpi=800, w=4*4, h=3*4, unit='in')
ggsave(plot=plot_price_enbs, filename=f"{enbs_paths}/04_price.pdf", dpi=800, w=4*4, h=3*4, unit='in')
ggsave(plot=plot_price_cycle_enbs, filename=f"{enbs_paths}/05_price_cycle.pdf", dpi=800, w=4*4, h=3*4, unit='in')

ggsave(plot=plot_1, filename=f"{enbs_paths}/05_main_results.pdf", dpi=800, w=4*4, h=3*4, unit='in')
ggsave(plot=plot_2, filename=f"{enbs_paths}/06_main_results_2.pdf", dpi=800, w=4*4, h=3*4, unit='in')
ggsave(plot=plot_3, filename=f"{enbs_paths}/07_main_results_3.pdf", dpi=800, w=4*4, h=3*4, unit='in')

'/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/03_enbs/07_main_results_3.pdf'